# Method comparison — same experiment, swappable steering method

A **shared, minimal** notebook so Charlotte and Arnav can compare methods on the *same* experiment
(generation tone). Everything is fixed and identical **except the one `METHOD` cell (§5)**.

**How to use.** Each person sets `METHOD_NAME` (or edits their row in `METHODS`), runs top to bottom,
and it saves `method_compare_<name>_<model>.json`. The last cell (§8) loads every such file and prints a
side-by-side comparison. Keep the model, OASIS images, prompts, and scorers identical so only the *method*
differs.

**What differs between methods** (from the repo notes): projection convention (mean-over-layers vs
last-token), which layers, and steering magnitude (norm-scaled `α` vs fixed coefficient). Those are exactly
the knobs in §5.

## 0 · Install

In [ ]:
!pip -q install transformers accelerate pillow numpy vaderSentiment

## 1 · Config (shared — keep identical across people)

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")  # reduce fragmentation (set before torch inits CUDA)
import json, contextlib, math
import numpy as np, torch
from PIL import Image

MODEL   = "google/gemma-3-4b-it"      # method comparison: same LIGHT model both can fit (matches Arnav). Switch to 12B once the method is chosen.
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16
OUT_DIR = "/content/out"; os.makedirs(OUT_DIR, exist_ok=True)
N_IMG   = 12          # small by default — this is for method comparison, not final numbers
N_PROMPT= 12
GEN_LEN = 40
TEMP    = 0.0
IMG_MAXDIM = 512
SEED    = 0
OASIS_IMG = "/content/affect_data/oasis_images"
OASIS_CSV = "/content/affect_data/OASIS.csv"
AFFECT_DIR = "/content/affect_data"   # pre-split folders here, or a Drive path e.g. /content/drive/MyDrive/affect_refusal/data
CANDS = {"lo":["images_negative","negative","distress","lo"],
         "mid":["images_neutral","neutral","mid"],
         "hi":["images_benign_emotional","images_positive","positive","benign_emotional","hi"]}

# >>> SET THIS to your method name (defined in §5) <<<
METHOD_NAME = "charlotte"
print("config ready |", MODEL, "| method:", METHOD_NAME)

## 1a · HF auth

In [ ]:
try:
    from huggingface_hub import login
    _t=os.environ.get("HF_TOKEN")
    if not _t:
        try:
            from google.colab import userdata; _t=userdata.get("HF_TOKEN")
        except Exception: _t=None
    if _t: login(_t); print("HF auth ok")
    else: print("!! set HF_TOKEN if the model 401s")
except Exception as e: print("auth note:", e)

## 2 · OASIS images (shared)

In [ ]:
import csv as _csv
def _load(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM:
        s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _from_split(key):                                                     # pre-split folders (e.g. your Drive layout)
    for name in CANDS[key]:
        d=os.path.join(AFFECT_DIR,name)
        if os.path.isdir(d):
            fs=[os.path.join(d,f) for f in sorted(os.listdir(d)) if f.lower().endswith((".jpg",".jpeg",".png",".webp"))]
            if fs: return [_load(p) for p in fs[:N_IMG]]
    return None
def load_oasis():
    if all(v in globals() for v in ("img_lo","img_mid","img_hi")):        # (a) reuse a split already in this kernel
        return dict(lo=img_lo[:N_IMG], mid=img_mid[:N_IMG], hi=img_hi[:N_IMG])
    lo,mid,hi=_from_split("lo"),_from_split("mid"),_from_split("hi")       # (b) pre-split folders under AFFECT_DIR
    if lo and hi: return dict(lo=lo, mid=(mid or lo), hi=hi)
    if os.path.isdir(OASIS_IMG) and os.path.isfile(OASIS_CSV):            # (c) raw OASIS + valence tertiles
        rows=[]
        with open(OASIS_CSV, newline="", encoding="utf-8", errors="ignore") as f:
            for r in _csv.DictReader(f):
                k={kk.strip():vv for kk,vv in r.items()}
                def g(*ns):
                    for n in ns:
                        for kk in k:
                            if kk.lower()==n: return k[kk]
                th=g("theme","file","filename","image"); v=g("valence_mean","valence")
                if not th or v is None: continue
                cand=[fn for fn in os.listdir(OASIS_IMG) if os.path.splitext(fn)[0].lower().startswith(str(th).strip().lower())]
                if cand:
                    try: rows.append((os.path.join(OASIS_IMG,cand[0]), float(v)))
                    except: pass
        rows.sort(key=lambda x:x[1]); t=len(rows)//3
        return dict(lo=[_load(p) for p,_ in rows[:t][:N_IMG]], mid=[_load(p) for p,_ in rows[t:2*t][:N_IMG]], hi=[_load(p) for p,_ in rows[-t:][-N_IMG:]])
    raise FileNotFoundError("No affect images. (a) run a notebook that defines img_lo/mid/hi in THIS kernel; "
        "(b) set AFFECT_DIR to a folder with pre-split subfolders (negative/neutral/positive); or (c) set OASIS_IMG + OASIS_CSV.")
IMGS=load_oasis(); print("images:", {k:len(v) for k,v in IMGS.items()})

## 3 · Prompts (shared, neutral)

In [ ]:
PROMPTS=[
 "Write a few sentences about an ordinary afternoon.",
 "Continue this: 'The next morning, she opened the door and'",
 "Describe a walk through a city you have never seen.",
 "Write a short passage about a train arriving at a station.",
 "Continue this: 'He picked up the letter and began to read.'",
 "Describe a room that has been empty for a while.",
 "Write about the view from a window.",
 "Continue this: 'The road stretched on ahead, and'",
 "Describe the inside of an old bookshop.",
 "Write a few sentences about waiting for a bus.",
 "Describe a quiet street at dusk.",
 "Continue this: 'The phone rang twice, and then'",
][:N_PROMPT]
print(len(PROMPTS),"prompts")

## 4 · Model + helpers (shared)

In [ ]:
import gc
for _n in ["model","proc","_sent"]:                      # free any model already on the GPU in this kernel
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); print("free GB before load:", round(torch.cuda.mem_get_info()[0]/1e9,1))
from transformers import AutoProcessor
try: from transformers import AutoModelForImageTextToText as _AutoVLM
except Exception: from transformers import AutoModelForVision2Seq as _AutoVLM
proc=AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model=_AutoVLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map=DEVICE, trust_remote_code=True).eval()
tok=proc.tokenizer if hasattr(proc,"tokenizer") else proc
def _layers(m):
    best=None
    for _,mod in m.named_modules():
        if isinstance(mod,torch.nn.ModuleList) and len(mod)>=8 and any(("attn" in n.lower() or "attention" in n.lower()) for n,_ in mod[0].named_modules()): best=mod
    return best
layers=_layers(model); nL=len(layers)
def bi(text, image=None):
    content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
    pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
    inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
    return {k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in inp.items()}
U=lambda v:(v/v.norm().clamp_min(1e-6)).to(DEVICE,DTYPE)
def add_hook(vec,coef):
    u=U(vec)
    def h(m,i,o): return (o[0]+coef*u,)+tuple(o[1:]) if isinstance(o,tuple) else o+coef*u
    return h
@contextlib.contextmanager
def hk(hooks):
    hd=[layers[l].register_forward_hook(h) for (l,h) in hooks]
    try: yield
    finally:
        for x in hd: x.remove()
def RL_last(inp):
    with torch.no_grad(): out=model(**inp, output_hidden_states=True)
    hs=out.hidden_states[1:1+nL]; return torch.stack([h.float()[0,-1].cpu() for h in hs])
def gen(prompt, image=None, hooks=()):
    inp=bi(prompt,image); L=inp["input_ids"].shape[1]
    kw=dict(max_new_tokens=GEN_LEN, do_sample=(TEMP>0), pad_token_id=tok.eos_token_id)
    if TEMP>0: kw.update(temperature=TEMP, top_p=0.95)
    with torch.no_grad(), hk(hooks): out=model.generate(**inp, **kw)
    return proc.batch_decode(out[:, L:], skip_special_tokens=True)[0].replace("\n"," ").strip()
print("helpers ready | layers", nL)

## 5 · METHOD — the only cell that differs between people

Set `METHOD_NAME` in §1, then edit **your** row below with your exact choices.
- `layers`: `"gate"` = [8,20)  ·  `"late"` = last 30%  ·  or an explicit list like `[18]`.
- `proj`: `"mean"` (mean over layers) or `"last"` (last-token) — for the internal axis readout.
- `steer`: `"normscaled"` (magnitude = `alpha` × ‖resid‖) or `"coeff"` (fixed `coeff` on the unit direction).

In [ ]:
METHODS = {
 # Charlotte: mean-over-layers projection, multi-layer norm-scaled steer, ALPHA_JB=0.008, gate [8,20)
 "charlotte": dict(layers="gate", proj="mean", steer="normscaled", alpha=0.008),
 # Arnav: last-token projection, fixed coefficient 20  (EDIT to your exact layers/coeff)
 "arnav":     dict(layers="late", proj="last", steer="coeff",      coeff=20.0),
}
METHOD=METHODS[METHOD_NAME]

# affect axis a (image valence: distress minus positive), diff-in-means, per layer
def _mean_last(imgs, prompt="Describe what is happening in this image."):
    return torch.stack([RL_last(bi(prompt,im)) for im in imgs[:N_IMG]]).mean(0)
a_dir=(_mean_last(IMGS["lo"])-_mean_last(IMGS["hi"])); a_dir=a_dir/a_dir.norm(dim=-1,keepdim=True).clamp_min(1e-6)

# resolve which layers this method steers
if   METHOD["layers"]=="gate": LAYERS=[l for l in range(8,20) if l<nL]
elif METHOD["layers"]=="late": LAYERS=list(range(int(0.7*nL), nL))
else: LAYERS=[l for l in METHOD["layers"] if l<nL]
# per-layer residual norms (for norm-scaled steering)
with torch.no_grad(): _o=model(**bi(PROMPTS[0]), output_hidden_states=True)
norms=np.array([float(h[0,-1].float().norm()) for h in _o.hidden_states[1:1+nL]])

def steer_hooks(sign):     # sign +1 = toward NEGATIVE valence (+a); -1 = toward POSITIVE (-a)
    if METHOD["steer"]=="normscaled":
        return [(l, add_hook(a_dir[l], sign*METHOD["alpha"]*norms[l])) for l in LAYERS]
    return [(l, add_hook(a_dir[l], sign*float(METHOD["coeff"]))) for l in LAYERS]
print("METHOD:", METHOD_NAME, "|", METHOD, "| steering layers:", LAYERS[:3], "...", LAYERS[-1])

## 6 · Scorers (shared)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
_vader=SentimentIntensityAnalyzer()
def score_vader(t): return float(_vader.polarity_scores(t or ".")["compound"])
try:
    from transformers import pipeline
    _sent=pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", top_k=None, device=0 if DEVICE=="cuda" else -1)
    def score_roberta(t):
        d={x["label"].lower():x["score"] for x in _sent((t or ".")[:512])[0]}
        return float(d.get("positive",0)-d.get("negative",0))
except Exception as e:
    score_roberta=None; print("RoBERTa unavailable:", e)
def score_axis(t):     # internal readout uses THIS METHOD's projection convention
    if not t.strip(): return 0.0
    with torch.no_grad(): o=model(**bi(t), output_hidden_states=True)
    hs=o.hidden_states[1:1+nL]
    r=[(hs[l][0].float().mean(0) if METHOD["proj"]=="mean" else hs[l][0,-1].float()).cpu() for l in range(nL)]
    return -float(np.mean([float(r[l]@a_dir[l]) for l in range(nL)]))/1000.0
print("scorers ready")

## 7 · Run + save (per method)

In [ ]:
def score(t): return (score_vader(t), score_roberta(t) if score_roberta else float("nan"), score_axis(t))
def cond(label, hooks=(), imgs=None):
    A=[]
    if imgs is None:
        for p in PROMPTS: A.append(score(gen(p, hooks=hooks)))
    else:
        for im in imgs[:N_IMG]:
            for p in PROMPTS: A.append(score(gen(p, image=im)))
    A=np.array(A)
    return dict(label=label, n=len(A), vader=float(np.nanmean(A[:,0])), roberta=float(np.nanmean(A[:,1])),
                axis=float(np.nanmean(A[:,2])), raw=A)
C={}
for lab,kw in [("image_distress",dict(imgs=IMGS["lo"])),("image_positive",dict(imgs=IMGS["hi"])),
               ("steer_+a_neg",dict(hooks=steer_hooks(+1))),("steer_-a_pos",dict(hooks=steer_hooks(-1)))]:
    print("running", lab, "..."); C[lab]=cond(lab, **kw)
def boot(a,b,col,it=2000):
    rng=np.random.default_rng(SEED); da=a[:,col][~np.isnan(a[:,col])]; db=b[:,col][~np.isnan(b[:,col])]
    d=float(np.mean(da)-np.mean(db)); bs=[np.mean(rng.choice(da,len(da)))-np.mean(rng.choice(db,len(db))) for _ in range(it)]
    return d, float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5))
EFF={}
for nm,hi,lo in [("image_effect","image_positive","image_distress"),("steer_effect","steer_-a_pos","steer_+a_neg")]:
    EFF[nm]={s:dict(zip(("diff","ci_lo","ci_hi"), boot(C[hi]["raw"],C[lo]["raw"],col))) for col,s in [(0,"VADER"),(1,"RoBERTa"),(2,"axis")]}
print("\n%-14s %10s %10s %10s"%("effect","VADER","RoBERTa","axis"))
for nm in EFF:
    print("%-14s %+10.3f %+10.3f %+10.3f"%(nm, EFF[nm]["VADER"]["diff"], EFF[nm]["RoBERTa"]["diff"], EFF[nm]["axis"]["diff"]))
out=dict(method=METHOD_NAME, method_params=METHOD, model=MODEL, layers=LAYERS,
         conditions={k:{kk:vv for kk,vv in v.items() if kk!="raw"} for k,v in C.items()}, effects=EFF)
tag="%s_%s"%(METHOD_NAME, MODEL.split("/")[-1])
json.dump(out, open(f"{OUT_DIR}/method_compare_{tag}.json","w"), indent=2, default=float)
print("\nsaved -> method_compare_%s.json  (run again with the other METHOD_NAME, then §8)"%tag)

## 8 · Compare — side by side (run after both methods are saved)

In [ ]:
import glob
files=sorted(glob.glob(f"{OUT_DIR}/method_compare_*.json"))
runs=[json.load(open(f)) for f in files]
if len(runs)<2:
    print("Only %d method run(s) found. Run this notebook with each METHOD_NAME, then re-run this cell."%len(runs))
else:
    print("%-14s %-10s"%("","")+" ".join("%-22s"%r["method"] for r in runs))
    for eff in ("image_effect","steer_effect"):
        for sc in ("VADER","RoBERTa","axis"):
            row="%-14s %-10s"%(eff if sc=="VADER" else "", sc)
            for r in runs:
                e=r["effects"][eff][sc]; sig="*" if (e["ci_lo"]>0 or e["ci_hi"]<0) else " "
                row+=" %+8.3f [%+.2f,%+.2f]%s"%(e["diff"],e["ci_lo"],e["ci_hi"],sig)
            print(row)
    print("\nimage_effect is method-independent (shared) -> should match across methods.")
    print("steer_effect depends on the METHOD -> this is the comparison: which steering reproduces the image effect.")